In [1]:

%pip -q install google-cloud-bigquery pandas pyarrow openpyxl

In [4]:
auth.authenticate_user()

print("Google authentication completed.")

Google authentication completed.


In [5]:
PROJECT_ID = "sme-media-analytics"

client = bigquery.Client(project=PROJECT_ID)

print("Connected to Google BigQuery.")
print("Project:", PROJECT_ID)

Connected to Google BigQuery.
Project: sme-media-analytics


In [6]:
OUTPUT = Path("/content/media_analytics")
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Project folder created:")
print(OUTPUT)

Project folder created:
/content/media_analytics


In [7]:
test_query = """
SELECT
    1 AS connection_test
"""

test_result = client.query(test_query).to_dataframe(
    create_bqstorage_client=False
)

display(test_result)

,connection_test
0,1


In [8]:
test_query = """
SELECT
    event_date,
    event_name,
    platform
FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE
    _TABLE_SUFFIX = '20201101'
LIMIT 10
"""

test_data = client.query(
    test_query
).to_dataframe(
    create_bqstorage_client=False
)

display(test_data)

,event_date,event_name,platform
0,20201101,page_view,WEB
1,20201101,first_visit,WEB
2,20201101,user_engagement,WEB
3,20201101,session_start,WEB
4,20201101,user_engagement,WEB
5,20201101,user_engagement,WEB
6,20201101,first_visit,WEB
7,20201101,page_view,WEB
8,20201101,page_view,WEB
9,20201101,session_start,WEB


In [9]:
query = """
SELECT
    event_date,
    event_timestamp,
    event_name,
    user_pseudo_id,
    platform,
    device.category AS device_category,
    geo.country AS country,
    traffic_source.source AS acquisition_source,
    traffic_source.medium AS acquisition_medium,
    traffic_source.name AS acquisition_campaign,

    (
        SELECT ep.value.int_value
        FROM UNNEST(event_params) AS ep
        WHERE ep.key = 'ga_session_id'
        LIMIT 1
    ) AS session_id,

    (
        SELECT ep.value.int_value
        FROM UNNEST(event_params) AS ep
        WHERE ep.key = 'engagement_time_msec'
        LIMIT 1
    ) AS engagement_time_msec

FROM
    `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`

WHERE
    _TABLE_SUFFIX BETWEEN '20201101' AND '20201130'
"""

raw_df = client.query(
    query
).to_dataframe(
    create_bqstorage_client=False
)

print("Raw data successfully extracted.")
print(f"Rows: {len(raw_df):,}")
print(f"Columns: {len(raw_df.columns)}")

Raw data successfully extracted.
Rows: 1,472,712
Columns: 12


In [10]:
display(raw_df.head(10))

,event_date,event_timestamp,event_name,user_pseudo_id,platform,device_category,country,acquisition_source,acquisition_medium,acquisition_campaign,session_id,engagement_time_msec
0,20201107,1604730371419225,scroll,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,5139
1,20201107,1604730379692069,page_view,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,<NA>
2,20201107,1604730494383076,page_view,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,<NA>
3,20201107,1604730379692069,view_search_results,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,<NA>
4,20201107,1604729517935082,user_engagement,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,6299
5,20201107,1604730182510780,user_engagement,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,18638
6,20201107,1604730197699616,scroll,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,9092
7,20201107,1604729749715867,page_view,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,<NA>
8,20201107,1604729573861767,page_view,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,<NA>
9,20201107,1604730485931421,scroll,1033552.6644233006,WEB,desktop,United States,(data deleted),(data deleted),<Other>,3627905599,12886


In [11]:
column_description = pd.DataFrame({
    "column_name": raw_df.columns,
    "data_type": raw_df.dtypes.astype(str).values,
    "missing_values": raw_df.isna().sum().values,
    "unique_values": [
        raw_df[column].nunique(dropna=True)
        for column in raw_df.columns
    ]
})

display(column_description)

,column_name,data_type,missing_values,unique_values
0,event_date,object,0,30
1,event_timestamp,Int64,0,1035369
2,event_name,object,0,17
3,user_pseudo_id,object,0,79421
4,platform,object,0,1
5,device_category,object,0,3
6,country,object,0,109
7,acquisition_source,object,0,5
8,acquisition_medium,object,0,6
9,acquisition_campaign,object,0,5


In [12]:
raw_file = OUTPUT / "ga4_raw_events.csv"

raw_df.to_csv(
    raw_file,
    index=False
)

print("Raw dataset saved to:")
print(raw_file)

Raw dataset saved to:
/content/media_analytics/ga4_raw_events.csv


In [13]:
file_size_mb = raw_file.stat().st_size / (1024 * 1024)

print(f"Raw dataset size: {file_size_mb:.2f} MB")

Raw dataset size: 172.51 MB


In [14]:
duplicate_count = raw_df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(raw_df) * 100:.2f}%"
)

Exact duplicate rows: 0
Duplicate percentage: 0.00%


In [15]:
missing_summary = (
    raw_df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(raw_df)
    * 100
).round(2)

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage
engagement_time_msec,666414,45.25
event_date,0,0.00
event_name,0,0.00
event_timestamp,0,0.00
user_pseudo_id,0,0.00
platform,0,0.00
country,0,0.00
device_category,0,0.00
acquisition_source,0,0.00
acquisition_medium,0,0.00


In [16]:
categorical_columns = [
    "event_name",
    "platform",
    "device_category",
    "country",
    "acquisition_source",
    "acquisition_medium",
    "acquisition_campaign"
]

for column in categorical_columns:
    print("\n" + "=" * 70)
    print(f"{column}")
    print("=" * 70)

    print(
        raw_df[column]
        .value_counts(dropna=False)
        .head(20)
    )


event_name
event_name
page_view              453904
user_engagement        407196
scroll                 170855
view_item              148639
session_start          106585
first_visit             71773
view_promotion          66789
begin_checkout           9546
view_search_results      8881
add_to_cart              7674
add_shipping_info        7492
add_payment_info         5125
select_promotion         3012
select_item              2369
purchase                 2054
click                     763
view_item_list             55
Name: count, dtype: int64

platform
platform
WEB    1472712
Name: count, dtype: int64

device_category
device_category
desktop    849334
mobile     590494
tablet      32884
Name: count, dtype: int64

country
country
United States     658643
India             135100
Canada            108228
United Kingdom     45307
Spain              29516
France             28025
Germany            24777
China              24638
Taiwan             23140
Italy              21107
J

In [17]:
raw_df["engagement_time_msec"].describe()

,engagement_time_msec
count,806298.0
mean,11803.425613
std,120633.861423
min,1.0
25%,1243.0
50%,5920.0
75%,11766.0
max,70799011.0


In [18]:
session_summary = pd.DataFrame({
    "total_rows": [len(raw_df)],
    "missing_session_ids": [
        raw_df["session_id"].isna().sum()
    ],
    "unique_session_ids": [
        raw_df["session_id"].nunique(dropna=True)
    ]
})

session_summary["missing_session_percentage"] = (
    session_summary["missing_session_ids"]
    / session_summary["total_rows"]
    * 100
).round(2)

display(session_summary)

,total_rows,missing_session_ids,unique_session_ids,missing_session_percentage
0,1472712,0,105380,0.0


In [19]:
user_summary = pd.DataFrame({
    "total_events": [len(raw_df)],
    "unique_users": [
        raw_df["user_pseudo_id"].nunique(dropna=True)
    ],
    "missing_user_ids": [
        raw_df["user_pseudo_id"].isna().sum()
    ]
})

display(user_summary)

,total_events,unique_users,missing_user_ids
0,1472712,79421,0


In [20]:
quality_report = pd.DataFrame({
    "column_name": raw_df.columns,
    "data_type": [
        str(raw_df[column].dtype)
        for column in raw_df.columns
    ],
    "row_count": [
        len(raw_df)
        for _ in raw_df.columns
    ],
    "missing_count": [
        raw_df[column].isna().sum()
        for column in raw_df.columns
    ],
    "missing_percentage": [
        round(raw_df[column].isna().mean() * 100, 2)
        for column in raw_df.columns
    ],
    "unique_values": [
        raw_df[column].nunique(dropna=True)
        for column in raw_df.columns
    ]
})

display(quality_report)

,column_name,data_type,row_count,missing_count,missing_percentage,unique_values
0,event_date,object,1472712,0,0.00,30
1,event_timestamp,Int64,1472712,0,0.00,1035369
2,event_name,object,1472712,0,0.00,17
3,user_pseudo_id,object,1472712,0,0.00,79421
4,platform,object,1472712,0,0.00,1
5,device_category,object,1472712,0,0.00,3
6,country,object,1472712,0,0.00,109
7,acquisition_source,object,1472712,0,0.00,5
8,acquisition_medium,object,1472712,0,0.00,6
9,acquisition_campaign,object,1472712,0,0.00,5


In [21]:
quality_file = OUTPUT / "data_quality_raw.csv"

quality_report.to_csv(
    quality_file,
    index=False
)

print("Data-quality report saved:")
print(quality_file)

Data-quality report saved:
/content/media_analytics/data_quality_raw.csv


In [22]:
clean_df = raw_df.copy()

print("Raw dataset remains unchanged.")
print(f"Clean dataset rows before cleaning: {len(clean_df):,}")

Raw dataset remains unchanged.
Clean dataset rows before cleaning: 1,472,712


In [23]:
clean_df["event_date"] = pd.to_datetime(
    clean_df["event_date"].astype("string"),
    format="%Y%m%d",
    errors="coerce"
)

print(clean_df["event_date"].dtype)

datetime64[ns]


In [24]:
clean_df["event_timestamp"] = pd.to_datetime(
    pd.to_numeric(
        clean_df["event_timestamp"],
        errors="coerce"
    ),
    unit="us",
    utc=True,
    errors="coerce"
)

print(clean_df["event_timestamp"].dtype)

datetime64[us, UTC]


In [25]:
text_columns = [
    "event_name",
    "platform",
    "device_category",
    "country",
    "acquisition_source",
    "acquisition_medium",
    "acquisition_campaign"
]

for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
    )

    clean_df[column] = clean_df[column].replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA
        }
    )

print("Text fields standardized.")

Text fields standardized.


In [26]:
clean_df["engagement_time_msec"] = pd.to_numeric(
    clean_df["engagement_time_msec"],
    errors="coerce"
)

In [27]:
negative_engagement = (
    clean_df["engagement_time_msec"] < 0
).sum()

print(
    f"Negative engagement values found: "
    f"{negative_engagement:,}"
)

Negative engagement values found: 0


In [28]:
clean_df.loc[
    clean_df["engagement_time_msec"] < 0,
    "engagement_time_msec"
] = np.nan

In [29]:
clean_df["engagement_seconds"] = (
    clean_df["engagement_time_msec"] / 1000
)

In [30]:
medium_to_channel = {
    "organic": "Organic Search",
    "cpc": "Paid Search",
    "ppc": "Paid Search",
    "referral": "Referral",
    "email": "Email",
    "social": "Social",
    "(none)": "Direct",
    "none": "Direct"
}

clean_df["medium_normalized"] = (
    clean_df["acquisition_medium"]
    .astype("string")
    .str.lower()
    .str.strip()
)

clean_df["channel"] = (
    clean_df["medium_normalized"]
    .map(medium_to_channel)
    .fillna("Other / Unknown")
)

print(
    clean_df["channel"]
    .value_counts(dropna=False)
)

channel
Organic Search     496289
Direct             334787
Other / Unknown    313256
Referral           267501
Paid Search         60879
Name: count, dtype: int64


In [31]:
clean_df["session_id"] = pd.to_numeric(
    clean_df["session_id"],
    errors="coerce"
).astype("Int64")

In [32]:
clean_df["session_key"] = (
    clean_df["user_pseudo_id"].astype("string")
    + "_"
    + clean_df["session_id"].astype("string")
)

In [33]:
invalid_session_mask = (
    clean_df["user_pseudo_id"].isna()
    | clean_df["session_id"].isna()
)

clean_df.loc[
    invalid_session_mask,
    "session_key"
] = pd.NA

In [34]:
assert clean_df["event_date"].notna().all(), (
    "Some event dates could not be converted."
)

assert clean_df["event_timestamp"].notna().all(), (
    "Some timestamps could not be converted."
)

assert (
    clean_df["engagement_time_msec"].dropna() >= 0
).all(), (
    "Negative engagement values remain."
)

assert (
    clean_df["engagement_seconds"].dropna() >= 0
).all(), (
    "Negative engagement seconds remain."
)

print("All validation checks passed.")

All validation checks passed.


In [35]:
cleaning_summary = pd.DataFrame({
    "metric": [
        "Raw rows",
        "Cleaned rows",
        "Rows removed",
        "Raw columns",
        "Cleaned columns",
        "Missing session IDs",
        "Unique users",
        "Unique sessions"
    ],
    "value": [
        len(raw_df),
        len(clean_df),
        len(raw_df) - len(clean_df),
        len(raw_df.columns),
        len(clean_df.columns),
        clean_df["session_id"].isna().sum(),
        clean_df["user_pseudo_id"].nunique(dropna=True),
        clean_df["session_key"].nunique(dropna=True)
    ]
})

display(cleaning_summary)

,metric,value
0,Raw rows,1472712
1,Cleaned rows,1472712
2,Rows removed,0
3,Raw columns,12
4,Cleaned columns,16
5,Missing session IDs,0
6,Unique users,79421
7,Unique sessions,108401


In [36]:
clean_file = OUTPUT / "ga4_cleaned_events.csv"

clean_df.to_csv(
    clean_file,
    index=False
)

print("Cleaned dataset saved:")
print(clean_file)

Cleaned dataset saved:
/content/media_analytics/ga4_cleaned_events.csv


In [37]:
valid_sessions = clean_df.dropna(
    subset=["session_key"]
).copy()

print(
    f"Events with valid session keys: "
    f"{len(valid_sessions):,}"
)

Events with valid session keys: 1,472,712


In [38]:
session_df = (
    valid_sessions
    .groupby("session_key", as_index=False)
    .agg(
        date=("event_date", "min"),
        channel=("channel", "first"),
        device=("device_category", "first"),
        country=("country", "first"),
        user_id=("user_pseudo_id", "first"),
        event_count=("event_name", "size"),
        engagement_seconds=(
            "engagement_seconds",
            lambda x: x.sum(min_count=1)
        ),
        has_user_engagement=(
            "event_name",
            lambda x: x.eq("user_engagement").any()
        )
    )
)

print(f"Session records created: {len(session_df):,}")

display(session_df.head())

Session records created: 108,401


,session_key,date,channel,device,country,user_id,event_count,engagement_seconds,has_user_engagement
0,1000300.3223254235_3614622791,2020-11-04,Referral,desktop,France,1000300.3223254235,2,<NA>,True
1,1000300.3223254235_9350310735,2020-11-04,Other / Unknown,desktop,France,1000300.3223254235,4,1.85,False
2,1000631.1195930056_2538888316,2020-11-06,Other / Unknown,mobile,(not set),1000631.1195930056,24,57.093,True
3,10006684.5628039063_2172078423,2020-11-13,Organic Search,tablet,United States,10006684.5628039063,11,51.867,True
4,10007461.3508636884_4575147746,2020-11-19,Organic Search,desktop,United States,10007461.3508636884,21,118.031,True


In [40]:
session_df["engagement_seconds"] = (
    pd.to_numeric(
        session_df["engagement_seconds"],
        errors="coerce"
    )
    .fillna(0)
)

session_df["has_user_engagement"] = (
    session_df["has_user_engagement"]
    .fillna(False)
    .astype(bool)
)

session_df["event_count"] = (
    pd.to_numeric(
        session_df["event_count"],
        errors="coerce"
    )
    .fillna(0)
)


session_df["engaged_session"] = (
    session_df["has_user_engagement"]
    | session_df["engagement_seconds"].ge(10)
    | session_df["event_count"].ge(2)
).astype("int8")

print("Engagement indicator created successfully.")

print(
    session_df["engaged_session"]
    .value_counts()
    .sort_index()
)

Engagement indicator created successfully.
engaged_session
0        82
1    108319
Name: count, dtype: int64


In [41]:
audience_daily = (
    session_df
    .groupby(
        ["date", "channel"],
        as_index=False
    )
    .agg(
        sessions=("session_key", "nunique"),
        users=("user_id", "nunique"),
        engaged_sessions=("engaged_session", "sum"),
        engagement_seconds=(
            "engagement_seconds",
            "sum"
        )
    )
)

audience_daily["engagement_rate"] = (
    audience_daily["engaged_sessions"]
    / audience_daily["sessions"]
).fillna(0)

display(audience_daily.head(10))

,date,channel,sessions,users,engaged_sessions,engagement_seconds,engagement_rate
0,2020-11-01,Direct,618,598,618,44228.45,1.000000
1,2020-11-01,Organic Search,918,898,918,52597.283,1.000000
2,2020-11-01,Other / Unknown,503,489,503,36688.405,1.000000
3,2020-11-01,Paid Search,108,107,108,7680.676,1.000000
4,2020-11-01,Referral,478,464,477,110566.425,0.997908
5,2020-11-02,Direct,832,811,830,63486.855,0.997596
6,2020-11-02,Organic Search,1216,1171,1214,89106.358,0.998355
7,2020-11-02,Other / Unknown,811,774,809,61403.561,0.997534
8,2020-11-02,Paid Search,176,176,176,13494.884,1.000000
9,2020-11-02,Referral,691,662,690,46052.313,0.998553


In [42]:
assert (
    audience_daily["engaged_sessions"]
    <= audience_daily["sessions"]
).all()

assert (
    audience_daily["engagement_rate"]
    >= 0
).all()

assert (
    audience_daily["engagement_rate"]
    <= 1
).all()

assert (
    audience_daily["sessions"]
    >= 0
).all()

print("Audience table validation passed.")

Audience table validation passed.


In [43]:
audience_file = OUTPUT / "audience_daily.csv"

audience_daily.to_csv(
    audience_file,
    index=False
)

print("Audience table saved:")
print(audience_file)

Audience table saved:
/content/media_analytics/audience_daily.csv


In [44]:
rng = np.random.default_rng(42)

advertisers = [
    "Northstar Retail",
    "Horizon Travel",
    "Summit Finance",
    "Evergreen Foods",
    "BluePeak Technology",
    "Urban Living"
]

campaign_types = [
    "Display",
    "Video",
    "Sponsored Content",
    "Social"
]

campaigns = pd.DataFrame({
    "campaign_id": [
        f"CMP-{i:03d}"
        for i in range(1, 19)
    ],
    "advertiser": np.repeat(
        advertisers,
        3
    ),
    "campaign_type": np.tile(
        campaign_types[:3],
        6
    )
})

campaigns["contract_value"] = rng.integers(
    15000,
    50000,
    size=len(campaigns)
).astype(float)

campaigns["contracted_cpm"] = rng.uniform(
    10,
    25,
    size=len(campaigns)
).round(2)

campaigns["start_date"] = pd.Timestamp(
    "2020-11-01"
)

campaigns["end_date"] = pd.Timestamp(
    "2020-11-30"
)

display(campaigns)

,campaign_id,advertiser,campaign_type,contract_value,contracted_cpm,start_date,end_date
0,CMP-001,Northstar Retail,Display,18123.0,16.76,2020-11-01,2020-11-30
1,CMP-002,Northstar Retail,Video,42088.0,15.56,2020-11-01,2020-11-30
2,CMP-003,Northstar Retail,Sponsored Content,37910.0,23.90,2020-11-01,2020-11-30
3,CMP-004,Horizon Travel,Display,30360.0,19.66,2020-11-01,2020-11-30
4,CMP-005,Horizon Travel,Video,30155.0,22.34,2020-11-01,2020-11-30
5,CMP-006,Horizon Travel,Sponsored Content,45050.0,16.65,2020-11-01,2020-11-30
6,CMP-007,Summit Finance,Display,18008.0,13.41,2020-11-01,2020-11-30
7,CMP-008,Summit Finance,Video,39407.0,18.32,2020-11-01,2020-11-30
8,CMP-009,Summit Finance,Sponsored Content,22051.0,10.96,2020-11-01,2020-11-30
9,CMP-010,Evergreen Foods,Display,18296.0,22.41,2020-11-01,2020-11-30


In [45]:
campaign_dates = pd.date_range(
    start="2020-11-01",
    end="2020-11-30",
    freq="D"
)

ad_delivery = pd.MultiIndex.from_product(
    [
        campaigns["campaign_id"],
        campaign_dates
    ],
    names=[
        "campaign_id",
        "date"
    ]
).to_frame(index=False)

ad_delivery = ad_delivery.merge(
    campaigns,
    on="campaign_id",
    how="left",
    validate="many_to_one"
)

print(
    f"Delivery records created: "
    f"{len(ad_delivery):,}"
)

display(ad_delivery.head())

Delivery records created: 540


,campaign_id,date,advertiser,campaign_type,contract_value,contracted_cpm,start_date,end_date
0,CMP-001,2020-11-01,Northstar Retail,Display,18123.0,16.76,2020-11-01,2020-11-30
1,CMP-001,2020-11-02,Northstar Retail,Display,18123.0,16.76,2020-11-01,2020-11-30
2,CMP-001,2020-11-03,Northstar Retail,Display,18123.0,16.76,2020-11-01,2020-11-30
3,CMP-001,2020-11-04,Northstar Retail,Display,18123.0,16.76,2020-11-01,2020-11-30
4,CMP-001,2020-11-05,Northstar Retail,Display,18123.0,16.76,2020-11-01,2020-11-30


In [46]:
impression_ranges = {
    "Display": (25000, 90000),
    "Video": (15000, 60000),
    "Sponsored Content": (10000, 45000)
}

def generate_impressions(campaign_type):
    low, high = impression_ranges[campaign_type]
    return rng.integers(low, high)

ad_delivery["impressions"] = (
    ad_delivery["campaign_type"]
    .map(generate_impressions)
    .astype(int)
)

display(
    ad_delivery[
        [
            "campaign_id",
            "campaign_type",
            "date",
            "impressions"
        ]
    ].head(10)
)

,campaign_id,campaign_type,date,impressions
0,CMP-001,Display,2020-11-01,57356
1,CMP-001,Display,2020-11-02,27847
2,CMP-001,Display,2020-11-03,60527
3,CMP-001,Display,2020-11-04,35028
4,CMP-001,Display,2020-11-05,73319
5,CMP-001,Display,2020-11-06,69398
6,CMP-001,Display,2020-11-07,84964
7,CMP-001,Display,2020-11-08,73409
8,CMP-001,Display,2020-11-09,48831
9,CMP-001,Display,2020-11-10,87888


In [47]:
ctr_ranges = {
    "Display": (0.005, 0.015),
    "Video": (0.008, 0.020),
    "Sponsored Content": (0.010, 0.025)
}

In [48]:
ad_delivery["simulated_ctr"] = ad_delivery[
    "campaign_type"
].map(
    lambda campaign_type: rng.uniform(
        *ctr_ranges[campaign_type]
    )
)

ad_delivery["clicks"] = (
    ad_delivery["impressions"]
    * ad_delivery["simulated_ctr"]
).round().astype(int)

In [49]:
assert (
    ad_delivery["clicks"]
    <= ad_delivery["impressions"]
).all()

assert (
    ad_delivery["clicks"]
    >= 0
).all()

print("Click validation passed.")

Click validation passed.


In [50]:
ad_delivery["revenue"] = (
    ad_delivery["impressions"]
    / 1000
    * ad_delivery["contracted_cpm"]
).round(2)

In [51]:
display(
    ad_delivery[
        [
            "campaign_id",
            "campaign_type",
            "impressions",
            "clicks",
            "contracted_cpm",
            "revenue"
        ]
    ].head(10)
)

,campaign_id,campaign_type,impressions,clicks,contracted_cpm,revenue
0,CMP-001,Display,57356,747,16.76,961.29
1,CMP-001,Display,27847,356,16.76,466.72
2,CMP-001,Display,60527,692,16.76,1014.43
3,CMP-001,Display,35028,448,16.76,587.07
4,CMP-001,Display,73319,465,16.76,1228.83
5,CMP-001,Display,69398,719,16.76,1163.11
6,CMP-001,Display,84964,862,16.76,1424.00
7,CMP-001,Display,73409,997,16.76,1230.33
8,CMP-001,Display,48831,470,16.76,818.41
9,CMP-001,Display,87888,778,16.76,1473.00


In [52]:
ad_delivery = ad_delivery.sort_values(
    [
        "campaign_id",
        "date"
    ]
).copy()

ad_delivery["cumulative_revenue"] = (
    ad_delivery
    .groupby("campaign_id")["revenue"]
    .cumsum()
)

In [53]:
previous_cumulative = (
    ad_delivery
    .groupby("campaign_id")["revenue"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

previous_cumulative = previous_cumulative.where(
    ad_delivery["campaign_id"]
    == ad_delivery["campaign_id"].shift(1),
    0
)

remaining_contract = (
    ad_delivery["contract_value"]
    - previous_cumulative
).clip(lower=0)

ad_delivery["recognized_revenue"] = (
    np.minimum(
        ad_delivery["revenue"],
        remaining_contract
    )
).round(2)

In [54]:
campaign_revenue_check = (
    ad_delivery
    .groupby("campaign_id", as_index=False)
    .agg(
        recognized_revenue=(
            "recognized_revenue",
            "sum"
        ),
        contract_value=(
            "contract_value",
            "first"
        )
    )
)

assert (
    campaign_revenue_check["recognized_revenue"]
    <= campaign_revenue_check["contract_value"]
    + 0.01
).all()

print("Contract revenue validation passed.")

Contract revenue validation passed.


In [55]:
ad_delivery["ctr"] = np.where(
    ad_delivery["impressions"] > 0,
    ad_delivery["clicks"]
    / ad_delivery["impressions"],
    0
)

In [56]:
ad_delivery["effective_cpm"] = np.where(
    ad_delivery["impressions"] > 0,
    ad_delivery["recognized_revenue"]
    * 1000
    / ad_delivery["impressions"],
    0
)

In [57]:
assert (
    ad_delivery["ctr"] >= 0
).all()

assert (
    ad_delivery["ctr"] <= 1
).all()

assert (
    ad_delivery["effective_cpm"] >= 0
).all()

print("Advertising KPI validation passed.")

Advertising KPI validation passed.


In [58]:
campaign_performance = (
    ad_delivery
    .groupby(
        [
            "campaign_id",
            "advertiser",
            "campaign_type"
        ],
        as_index=False
    )
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        revenue=("recognized_revenue", "sum"),
        contract_value=("contract_value", "first")
    )
)

In [59]:
campaign_performance["ctr"] = np.where(
    campaign_performance["impressions"] > 0,
    campaign_performance["clicks"]
    / campaign_performance["impressions"],
    0
)

In [60]:
campaign_performance["effective_cpm"] = np.where(
    campaign_performance["impressions"] > 0,
    campaign_performance["revenue"]
    * 1000
    / campaign_performance["impressions"],
    0
)

In [61]:
campaign_performance["contract_fulfillment"] = np.where(
    campaign_performance["contract_value"] > 0,
    campaign_performance["revenue"]
    / campaign_performance["contract_value"],
    0
)

In [62]:
assert (
    campaign_performance["revenue"]
    >= 0
).all()

assert (
    campaign_performance["revenue"]
    <= campaign_performance["contract_value"]
    + 0.01
).all()

assert (
    campaign_performance["contract_fulfillment"]
    >= 0
).all()

assert (
    campaign_performance["contract_fulfillment"]
    <= 1.001
).all()

print("Campaign performance validation passed.")

Campaign performance validation passed.


In [63]:
display(
    campaign_performance.sort_values(
        "revenue",
        ascending=False
    ).head(10)
)

,campaign_id,advertiser,campaign_type,impressions,clicks,revenue,contract_value,ctr,effective_cpm,contract_fulfillment
15,CMP-016,Urban Living,Display,1708792,16204,37046.63,42512.0,0.009483,21.680011,0.871439
3,CMP-004,Horizon Travel,Display,1702544,16946,30360.00,30360.0,0.009953,17.832138,1.000000
13,CMP-014,BluePeak Technology,Video,1163740,17682,28581.46,41639.0,0.015194,24.560005,0.686411
12,CMP-013,BluePeak Technology,Display,1711373,16236,26218.28,40751.0,0.009487,15.320027,0.643378
4,CMP-005,Horizon Travel,Video,1139086,15770,25447.20,30155.0,0.013844,22.340016,0.843880
10,CMP-011,Evergreen Foods,Video,1166869,18641,22718.94,33426.0,0.015975,19.470000,0.679679
14,CMP-015,BluePeak Technology,Sponsored Content,875880,13335,20495.60,40111.0,0.015225,23.400009,0.510972
7,CMP-008,Summit Finance,Video,1111725,15465,20366.83,39407.0,0.013911,18.320025,0.516833
1,CMP-002,Northstar Retail,Video,1206745,17214,18776.94,42088.0,0.014265,15.559990,0.446135
2,CMP-003,Northstar Retail,Sponsored Content,772019,13992,18451.25,37910.0,0.018124,23.899995,0.486712


In [64]:
campaign_file = OUTPUT / "campaigns.csv"

delivery_file = OUTPUT / "ad_delivery.csv"

performance_file = OUTPUT / "campaign_performance.csv"

campaigns.to_csv(
    campaign_file,
    index=False
)

ad_delivery.to_csv(
    delivery_file,
    index=False
)

campaign_performance.to_csv(
    performance_file,
    index=False
)

print("Advertising datasets saved successfully.")

Advertising datasets saved successfully.


In [65]:
database_path = OUTPUT / "media_analytics.db"

connection = sqlite3.connect(
    database_path
)

print("SQLite database created:")
print(database_path)

SQLite database created:
/content/media_analytics/media_analytics.db


In [66]:
audience_daily.to_sql(
    "audience_daily",
    connection,
    if_exists="replace",
    index=False
)

campaigns.to_sql(
    "campaigns",
    connection,
    if_exists="replace",
    index=False
)

ad_delivery.to_sql(
    "ad_delivery",
    connection,
    if_exists="replace",
    index=False
)

campaign_performance.to_sql(
    "campaign_performance",
    connection,
    if_exists="replace",
    index=False
)

print("All analytical tables loaded into SQLite.")

All analytical tables loaded into SQLite.


In [67]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection
)

display(tables)

,name
0,ad_delivery
1,audience_daily
2,campaign_performance
3,campaigns


In [68]:
revenue_by_advertiser = pd.read_sql_query(
    """
    SELECT
        advertiser,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM campaign_performance
    GROUP BY advertiser
    ORDER BY total_revenue DESC
    """,
    connection
)

display(revenue_by_advertiser)

,advertiser,total_revenue
0,BluePeak Technology,75295.34
1,Horizon Travel,69529.55
2,Urban Living,65367.39
3,Evergreen Foods,57019.10
4,Northstar Retail,55351.19
5,Summit Finance,47219.33


In [69]:
revenue_by_type = pd.read_sql_query(
    """
    SELECT
        campaign_type,
        ROUND(SUM(revenue), 2) AS total_revenue,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks
    FROM campaign_performance
    GROUP BY campaign_type
    ORDER BY total_revenue DESC
    """,
    connection
)

display(revenue_by_type)

,campaign_type,total_revenue,impressions,clicks
0,Display,148051.91,10169908,100179
1,Video,130100.73,6887960,99704
2,Sponsored Content,91629.26,4858035,83408


In [70]:
campaign_kpis_sql = pd.read_sql_query(
    """
    SELECT
        campaign_id,
        advertiser,
        campaign_type,

        impressions,

        clicks,

        ROUND(revenue, 2) AS revenue,

        ROUND(
            100.0 * clicks
            / NULLIF(impressions, 0),
            2
        ) AS ctr_percent,

        ROUND(
            revenue * 1000.0
            / NULLIF(impressions, 0),
            2
        ) AS effective_cpm

    FROM campaign_performance

    ORDER BY revenue DESC
    """,
    connection
)

display(campaign_kpis_sql.head(10))

,campaign_id,advertiser,campaign_type,impressions,clicks,revenue,ctr_percent,effective_cpm
0,CMP-016,Urban Living,Display,1708792,16204,37046.63,0.95,21.68
1,CMP-004,Horizon Travel,Display,1702544,16946,30360.00,1.00,17.83
2,CMP-014,BluePeak Technology,Video,1163740,17682,28581.46,1.52,24.56
3,CMP-013,BluePeak Technology,Display,1711373,16236,26218.28,0.95,15.32
4,CMP-005,Horizon Travel,Video,1139086,15770,25447.20,1.38,22.34
5,CMP-011,Evergreen Foods,Video,1166869,18641,22718.94,1.60,19.47
6,CMP-015,BluePeak Technology,Sponsored Content,875880,13335,20495.60,1.52,23.40
7,CMP-008,Summit Finance,Video,1111725,15465,20366.83,1.39,18.32
8,CMP-002,Northstar Retail,Video,1206745,17214,18776.94,1.43,15.56
9,CMP-003,Northstar Retail,Sponsored Content,772019,13992,18451.25,1.81,23.90


In [71]:
reconciliation = pd.read_sql_query(
    """
    SELECT
        (
            SELECT ROUND(
                SUM(recognized_revenue),
                2
            )
            FROM ad_delivery
        ) AS delivery_revenue,

        (
            SELECT ROUND(
                SUM(revenue),
                2
            )
            FROM campaign_performance
        ) AS summary_revenue
    """,
    connection
)

display(reconciliation)

,delivery_revenue,summary_revenue
0,369781.9,369781.9


In [72]:
reconciliation["difference"] = (
    reconciliation["delivery_revenue"]
    - reconciliation["summary_revenue"]
)

display(reconciliation)

,delivery_revenue,summary_revenue,difference
0,369781.9,369781.9,0.0


In [73]:
assert (
    abs(reconciliation["difference"].iloc[0])
    < 0.01
)

print("Revenue reconciliation passed.")

Revenue reconciliation passed.


In [74]:
connection.execute(
    """
    DROP VIEW IF EXISTS campaign_reporting
    """
)

connection.execute(
    """
    CREATE VIEW campaign_reporting AS

    SELECT
        campaign_id,
        advertiser,
        campaign_type,
        impressions,
        clicks,
        revenue,
        contract_value,

        ROUND(
            100.0 * clicks
            / NULLIF(impressions, 0),
            2
        ) AS ctr_percent,

        ROUND(
            revenue * 1000.0
            / NULLIF(impressions, 0),
            2
        ) AS effective_cpm,

        ROUND(
            100.0 * revenue
            / NULLIF(contract_value, 0),
            2
        ) AS contract_fulfillment_percent

    FROM campaign_performance
    """
)

connection.commit()

print("SQL reporting view created.")

SQL reporting view created.


In [75]:
reporting_table = pd.read_sql_query(
    """
    SELECT *
    FROM campaign_reporting
    ORDER BY revenue DESC
    """,
    connection
)

display(reporting_table.head(10))

,campaign_id,advertiser,campaign_type,impressions,clicks,revenue,contract_value,ctr_percent,effective_cpm,contract_fulfillment_percent
0,CMP-016,Urban Living,Display,1708792,16204,37046.63,42512.0,0.95,21.68,87.14
1,CMP-004,Horizon Travel,Display,1702544,16946,30360.00,30360.0,1.00,17.83,100.00
2,CMP-014,BluePeak Technology,Video,1163740,17682,28581.46,41639.0,1.52,24.56,68.64
3,CMP-013,BluePeak Technology,Display,1711373,16236,26218.28,40751.0,0.95,15.32,64.34
4,CMP-005,Horizon Travel,Video,1139086,15770,25447.20,30155.0,1.38,22.34,84.39
5,CMP-011,Evergreen Foods,Video,1166869,18641,22718.94,33426.0,1.60,19.47,67.97
6,CMP-015,BluePeak Technology,Sponsored Content,875880,13335,20495.60,40111.0,1.52,23.40,51.10
7,CMP-008,Summit Finance,Video,1111725,15465,20366.83,39407.0,1.39,18.32,51.68
8,CMP-002,Northstar Retail,Video,1206745,17214,18776.94,42088.0,1.43,15.56,44.61
9,CMP-003,Northstar Retail,Sponsored Content,772019,13992,18451.25,37910.0,1.81,23.90,48.67


In [76]:
business_summary = pd.read_sql_query(
    """
    SELECT
        advertiser,

        ROUND(
            SUM(revenue),
            2
        ) AS revenue,

        SUM(impressions) AS impressions,

        SUM(clicks) AS clicks,

        ROUND(
            100.0 * SUM(clicks)
            / NULLIF(SUM(impressions), 0),
            2
        ) AS ctr_percent,

        ROUND(
            SUM(revenue) * 1000.0
            / NULLIF(SUM(impressions), 0),
            2
        ) AS effective_cpm,

        ROUND(
            100.0 * SUM(revenue)
            / NULLIF(SUM(contract_value), 0),
            2
        ) AS contract_fulfillment_percent

    FROM campaign_performance

    GROUP BY advertiser

    ORDER BY revenue DESC
    """,
    connection
)

display(business_summary)

,advertiser,revenue,impressions,clicks,ctr_percent,effective_cpm,contract_fulfillment_percent
0,BluePeak Technology,75295.34,3750993,47253,1.26,20.07,61.47
1,Horizon Travel,69529.55,3665797,46749,1.28,18.97,65.86
2,Urban Living,65367.39,3638670,45861,1.26,17.96,68.84
3,Evergreen Foods,57019.10,3624248,48869,1.35,15.73,56.53
4,Northstar Retail,55351.19,3712286,48855,1.32,14.91,56.41
5,Summit Finance,47219.33,3523909,45704,1.30,13.40,59.42


In [77]:
connection.close()

print("SQLite connection closed successfully.")

SQLite connection closed successfully.


In [78]:
powerbi_file = OUTPUT / "SME_Media_Performance_PowerBI.xlsx"

with pd.ExcelWriter(
    powerbi_file,
    engine="openpyxl"
) as writer:

    audience_daily.to_excel(
        writer,
        sheet_name="AudienceDaily",
        index=False
    )

    campaigns.to_excel(
        writer,
        sheet_name="Campaigns",
        index=False
    )

    ad_delivery.to_excel(
        writer,
        sheet_name="AdDelivery",
        index=False
    )

    campaign_performance.to_excel(
        writer,
        sheet_name="CampaignPerformance",
        index=False
    )

    business_summary.to_excel(
        writer,
        sheet_name="BusinessSummary",
        index=False
    )

print("Power BI workbook created:")
print(powerbi_file)

Power BI workbook created:
/content/media_analytics/SME_Media_Performance_PowerBI.xlsx


In [79]:
print(
    f"Workbook size: "
    f"{powerbi_file.stat().st_size / 1024:.1f} KB"
)

Workbook size: 69.8 KB


In [80]:
from google.colab import files

files.download(
    str(powerbi_file)
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>